# Controller 集成验证
历史样本→MySQL→显式数据准备→纯计算器→计算记录。组合类的示例使用历史模拟 ES。

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from data import Database, MySQLConfig
from data import codec
database = Database(MySQLConfig.load())
database.health()

{'database_name': 'FRM', 'server_version': '9.7.2', 'schema_version': 1}

In [2]:
from controller import RiskController
from service import load_chapter
chapter = load_chapter(1)
frame = pd.DataFrame({'r': [-0.04, -0.02, 0.0, 0.01, 0.03, np.nan]})
def historical_es(df, target_column, confidence_level):
    return chapter.ExpectedShortfall(
        chapter.HistoricalLossQuantile(df, target_column), confidence_level
    ).calculate()

dataset_id = run_id = es_id = None
try:
    with database.session() as repository:
        dataset_id = repository.save_dataset('controller-integration-test', frame)
        controller = RiskController(repository)
        result = controller.calculate(dataset_id, 1, 'cal_normal_var',
            parameters={'target_column':'r', 'confidence_level':0.95},
            preparation={'dropna_columns':['r']})
        run_id = result.run_id
        np.testing.assert_allclose(result.value, chapter.cal_normal_var(frame.dropna(), target_column='r'))
        es = controller.run(dataset_id, 'historical_es', historical_es,
            parameters={'target_column':'r', 'confidence_level':0.8},
            preparation={'dropna_columns':['r']})
        es_id = es.run_id
        np.testing.assert_allclose(es.value, 0.04)
        assert controller.calculate(dataset_id, 1, 'cal_normal_var',
            parameters={'target_column':'r'}, save=False).run_id is None
        pd.testing.assert_frame_equal(repository.load_dataset(dataset_id).frame, frame)
    with database.session() as repository:
        stored = repository.load_calculation(run_id)
        np.testing.assert_equal(stored['result'], result.value)
        assert stored['parameters']['prepared_rows'] == 5
        assert len(repository.list_calculations(dataset_id)) == 2
    print('函数、类组合、计算记录跨连接读取及原始样本不变：通过')
finally:
    if dataset_id:
        with database.session() as repository:
            with repository.connection.cursor() as cursor:
                cursor.execute('DELETE FROM frm_calculations WHERE dataset_id=%s', (dataset_id,))
                cursor.execute('DELETE FROM frm_datasets WHERE dataset_id=%s', (dataset_id,))

函数、类组合、计算记录跨连接读取及原始样本不变：通过


In [3]:
for number in range(1, 19):
    assert load_chapter(number) is not None
print('18章统一加载通过')

18章统一加载通过
